# 🚀 Zyro Dynamics HR Help Desk — RAG Challenge
**Optimized Submission | BGE-base Embeddings | Multi-Query MMR | LangSmith Tracing**

Scoring: Q01–Q10 (in-scope, 8 pts each) + Q11–Q15 (out-of-scope refusal, 4 pts each) = 100 pts

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q langchain langchain-community langchain-text-splitters \
    langchain-huggingface langchain-groq langchain-core langsmith \
    faiss-cpu pypdf sentence-transformers huggingface_hub rank_bm25
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Imports & API keys ────────────────────────────────────────────────
import os, re, base64
import pandas as pd
from kaggle_secrets import UserSecretsClient

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

secrets = UserSecretsClient()
os.environ['GROQ_API_KEY']         = secrets.get_secret('GROQ_API_KEY')
os.environ['LANGCHAIN_API_KEY']    = secrets.get_secret('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']    = 'zyro-rag-challenge'

print('✅ API keys loaded')
print(f'   GROQ key: ...{os.environ["GROQ_API_KEY"][-6:]}')
print(f'   LangSmith tracing: ON → project: zyro-rag-challenge')

In [ ]:
# ── Cell 3: TODO — Load HR PDFs ───────────────────────────────────────────────
import glob

# Auto-detect the PDF directory from Kaggle dataset
search_paths = [
    '/kaggle/input/zyro-dynamics-hr-policies',
    '/kaggle/input/zyro-hr-policies',
    '/kaggle/input',
]
pdf_files = []
DOCS_DIR  = None
for p in search_paths:
    found = glob.glob(f'{p}/**/*.pdf', recursive=True)
    if found:
        pdf_files = found
        DOCS_DIR  = p if os.path.isdir(p) else os.path.dirname(found[0])
        break

if not pdf_files:
    raise FileNotFoundError('No PDFs found! Check your Kaggle dataset is attached.')

print(f'📄 Found {len(pdf_files)} PDF files in: {DOCS_DIR}')
for f in sorted(pdf_files):
    print(f'   {os.path.basename(f)}')

loader    = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()
print(f'\n✅ Loaded {len(documents)} pages from {len(pdf_files)} documents')

In [ ]:
# ── Cell 4: TODO — Chunk documents + pre-extract full texts ───────────────────
# Pre-extract FULL TEXT per source document for parent-document retrieval
doc_full_texts = {}
for doc in documents:
    source = doc.metadata.get('source', '')
    if source not in doc_full_texts:
        doc_full_texts[source] = ''
    doc_full_texts[source] += doc.page_content + '\n\n'
print(f'📄 Pre-extracted full text for {len(doc_full_texts)} source documents')

# Chunk with smaller size for finer-grained retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', '! ', '? ', '; ', ', ', ' ', ''],
    add_start_index=True,
)
chunks = splitter.split_documents(documents)
print(f'✅ Created {len(chunks)} chunks  (size=500, overlap=150)')

In [ ]:
# ── Cell 5: TODO — Build hybrid retriever (BM25 + FAISS) ─────────────────────
print('⏳ Loading BAAI/bge-base-en-v1.5 embedding model…')
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# FAISS semantic retriever (MMR for diversity)
vectorstore = FAISS.from_documents(chunks, embeddings)
semantic_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 10, 'fetch_k': 40, 'lambda_mult': 0.5},
)

# BM25 keyword retriever (catches exact term matches embeddings miss)
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)

# Hybrid ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.3, 0.7],
)
print('✅ Hybrid retriever built (BM25 + FAISS MMR)')

In [ ]:
# ── Cell 6: TODO — Build LCEL RAG chain + guardrails (OPTIMIZED) ──────────────
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, max_tokens=2048)

# --- PRECISION-focused RAG prompt ---
rag_prompt = ChatPromptTemplate.from_template("""\
You are the HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Answer the employee's question using ONLY the policy documents provided below.

INSTRUCTIONS:
1. Use ONLY information from the Context. Never add external knowledge.
2. Be PRECISE: state exact numbers, days, percentages, amounts, dates, and eligibility criteria directly from the policy.
3. Be COMPLETE: include ALL relevant details, conditions, exceptions, and related rules.
4. Be DIRECT: start with the answer immediately. Do NOT add filler phrases like 'According to the policy' or 'As per the HR documents' or 'Based on the provided context'.
5. For questions about entitlements or limits, state the specific values clearly.
6. For questions about processes, list the steps in order.
7. For tabular data, present the key data points clearly.
8. Use bullet points for multi-part answers.
9. If the context does NOT contain the answer, respond ONLY with: 'This information is not available in the HR policy documents. Please contact HR directly.'

Context (from Zyro Dynamics HR Policy Documents):
{context}

Employee Question: {question}

Answer:""")

scope_prompt = ChatPromptTemplate.from_template("""\
You are a classifier. Determine if the following question is related to:
- Company HR policies, employment, workplace rules
- Leave, attendance, holidays
- Benefits, compensation, salary, insurance
- Workplace conduct, ethics, harassment
- IT security, data protection, devices
- Onboarding, separation, probation, retirement
- Travel expenses, reimbursements
- Performance management, reviews, promotions

Answer ONLY with one word: YES or NO.

Question: {question}
Answer:""")

multi_query_prompt = ChatPromptTemplate.from_template("""\
Generate 3 different phrasings of the following HR question to improve document retrieval.
Output ONLY the 3 questions, one per line, no numbering, no extra text.

Original question: {question}

3 rephrased questions:""")

# --- Standardized refusal message ---
REFUSAL_MSG = (
    "I can only answer questions related to Zyro Dynamics HR policies. "
    "This question is outside my scope. Please contact HR directly for non-policy queries."
)

# --- HR keyword sets (expanded) ---
HR_KEYWORDS = {
    'leave','vacation','pto','sick','maternity','paternity','wfh','remote','hybrid',
    'office','salary','ctc','bonus','increment','appraisal','performance','pip',
    'review','rating','probation','onboarding','offboarding','separation','resignation',
    'notice','travel','expense','reimbursement','policy','handbook','conduct',
    'harassment','posh','icc','device','security','data','confidential','employee',
    'hr','benefit','insurance','pf','gratuity','esic','holiday','overtime','shift',
    'allowance','deduction','tax','tds','grade','band','promotion','transfer','zyro',
    'dynamics','company','joining','full','final','settlement','termination',
    'disciplinary','code','ethics','misconduct','warning','annual','earned','casual',
    'compensatory','bereavement','accommodation','hotel','flight','cab','conveyance',
    'work','hours','attendance','payroll','stipend','internship',
    'retirement','okr','objectives','feedback','calibration',
    'encryption','laptop','password','vpn','firewall','antivirus',
    'marriage','wedding','encashment','carry','forward',
    'confirmation','induction','relieving','experience',
    'letter','exit','interview','handover','knowledge',
}

OUT_OF_SCOPE_KEYWORDS = {
    'weather','cricket','football','sports','movie','recipe','cook','stock','crypto',
    'bitcoin','politics','government','news','celebrity','music','song','gaming',
    'restaurant','astrology','horoscope','joke','poem','story','doctor',
    'medical diagnosis','investment','share market','lottery','betting',
    'capital','president','prime minister','planet','solar system',
    'programming','python','javascript','machine learning','ai model',
}

# --- Helpers ---
def is_hr_question(question):
    words = set(re.findall(r'\b\w+\b', question.lower()))
    if words & OUT_OF_SCOPE_KEYWORDS:
        return False
    if words & HR_KEYWORDS:
        return True
    return None

def multi_query_retrieve(question):
    """Generate query variants and retrieve with hybrid ensemble, deduplicated."""
    try:
        raw      = (multi_query_prompt | llm | StrOutputParser()).invoke({'question': question})
        variants = [q.strip() for q in raw.strip().split('\n') if q.strip()][:3]
    except Exception:
        variants = []
    seen_ids, docs = set(), []
    for q in [question] + variants:
        try:
            for doc in ensemble_retriever.invoke(q):
                doc_id = hash(doc.page_content[:200])
                if doc_id not in seen_ids:
                    seen_ids.add(doc_id)
                    docs.append(doc)
        except Exception:
            pass
    return docs or ensemble_retriever.invoke(question)

def get_full_document_context(question):
    """Parent-document retrieval: find relevant chunks, then inject
    FULL text of the top matching source documents as context."""
    initial_docs = multi_query_retrieve(question)
    if not initial_docs:
        return '', []
    # Count which source documents appear most
    source_counts = {}
    for doc in initial_docs:
        src = doc.metadata.get('source', '')
        source_counts[src] = source_counts.get(src, 0) + 1
    # Top 3 most-relevant source documents
    top_sources = sorted(source_counts, key=source_counts.get, reverse=True)[:3]
    # Build context from FULL document texts
    context_parts = []
    for src in top_sources:
        if src in doc_full_texts:
            context_parts.append(doc_full_texts[src])
    return '\n\n---\n\n'.join(context_parts), top_sources

# --- Main RAG function ---
def ask_bot(question: str) -> dict:
    question = question.strip()
    if not question:
        return {'answer': 'Please ask a question.', 'sources': []}

    # Layer 1: keyword fast-path
    keyword_result = is_hr_question(question)
    if keyword_result is False:
        return {'answer': REFUSAL_MSG, 'sources': []}

    # Layer 2: LLM scope classification for ambiguous questions
    if keyword_result is None:
        try:
            cls = (scope_prompt | llm | StrOutputParser()).invoke({'question': question}).strip().upper()
            if 'NO' in cls and 'YES' not in cls:
                return {'answer': REFUSAL_MSG, 'sources': []}
        except Exception:
            pass

    # Layer 3: Parent-document context retrieval
    context, sources = get_full_document_context(question)

    if not context.strip():
        return {
            'answer': 'This information is not available in the HR policy documents. Please contact HR directly.',
            'sources': []
        }

    # Layer 4: Generate precise answer with full document context
    chain = (
        {'context': RunnableLambda(lambda _: context), 'question': RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    )
    answer  = chain.invoke(question)
    return {'answer': answer, 'sources': sources}

print('✅ RAG chain with parent-doc retrieval + hybrid search ready')

In [ ]:
# ── Cell 7: TODO — Test guardrails ────────────────────────────────────────────
test_oos = ask_bot('What is the capital of France?')
print('Out-of-scope test:')
print(f'  Q: What is the capital of France?')
print(f'  A: {test_oos["answer"]}')
print()
assert 'outside my scope' in test_oos['answer'], 'Guardrail not working!'
print('✅ Guardrails working correctly')

In [ ]:
# ── Cell 8: Encoding helpers ─────────────────────────────────────────────────
def encode_text(text: str) -> str:
    return base64.b64encode(text.encode('utf-8')).decode('utf-8')

def decode_text(encoded: str) -> str:
    return base64.b64decode(encoded.encode('utf-8')).decode('utf-8')

print('✅ Encoding helpers ready')

In [ ]:
# ── Cell 9: PASTE ENCODED QUESTIONS HERE ─────────────────────────────────────
# Instructions:
# 1. Open the ORIGINAL starter notebook on Kaggle
# 2. Find the cell that defines the encoded questions (question_enc list)
# 3. Copy each encoded string and paste it below
# 4. The format is base64-encoded text

QUESTIONS_ENC = [
    # Paste your 15 encoded questions here as strings, in order Q01-Q15
    # Example:
    # 'V2hhdCBpcyB0aGUgYW5udWFsIGxlYXZlIHBvbGljeT8=',  # Q01
    # 'V2hhdCBpcyB0aGUgV0ZIIHBvbGljeT8=',              # Q02
    # ... and so on for Q03-Q15
]

# If you have them as a dict:
# QUESTIONS_ENC = {'Q01': '...', 'Q02': '...', ...}

if QUESTIONS_ENC:
    print(f'✅ {len(QUESTIONS_ENC)} encoded questions loaded')
    for i, enc in enumerate(QUESTIONS_ENC[:3], 1):
        print(f'  Q{i:02d}: {decode_text(enc)}')
else:
    print('⚠️  No questions loaded yet — paste encoded questions above')

In [ ]:
# ── Cell 10: Your Submission Links ───────────────────────────────────────────
STREAMLIT_URL = 'PASTE_YOUR_STREAMLIT_URL_HERE'
LANGSMITH_URL = 'PASTE_YOUR_LANGSMITH_TRACE_URL_HERE'

print(f'Streamlit : {STREAMLIT_URL}')
print(f'LangSmith : {LANGSMITH_URL}')

In [ ]:
# ── Cell 11: Run all 15 questions & collect answers ───────────────────────────
results = []

q_ids = [f'Q{i:02d}' for i in range(1, 16)]

for idx, q_enc in enumerate(QUESTIONS_ENC):
    qid      = q_ids[idx] if idx < len(q_ids) else f'Q{idx+1:02d}'
    question = decode_text(q_enc)

    print(f'\n{\'=\'*60}')
    print(f'🔍 {qid}: {question}')

    response   = ask_bot(question)
    answer     = response['answer']
    answer_enc = encode_text(answer)

    print(f'💬 {answer[:250]}...' if len(answer) > 250 else f'💬 {answer}')
    if response['sources']:
        src_names = [os.path.basename(s) for s in response['sources']]
        print(f'📄 Sources: {src_names}')

    results.append({
        'question_id':    qid,
        'question_enc':   q_enc,
        'answer_enc':     answer_enc,
        'streamlit_link': STREAMLIT_URL,
        'langsmith_link': LANGSMITH_URL,
    })

print(f'\n\n✅ Answered {len(results)} / 15 questions')

In [ ]:
# ── Cell 12: Sample test questions (as per competition Cell 12) ───────────────
print('\n📋 SAMPLE TEST QUESTIONS\n')
sample_questions = [
    'How many casual leaves are employees entitled to per year?',
    'What is the work from home policy for employees at Zyro Dynamics?',
    'What are the guidelines for business travel reimbursements?',
]
for q in sample_questions:
    print(f'Q: {q}')
    res = ask_bot(q)
    print(f'A: {res["answer"]}')
    print()

In [ ]:
# ── Cell 13: LangSmith trace URL instructions ─────────────────────────────────
print("""
📡 HOW TO GET YOUR LANGSMITH TRACE URL
======================================
1. Go to https://smith.langchain.com
2. Sign in → click 'Projects' in left sidebar
3. Open project: zyro-rag-challenge
4. Click on any trace → click 'Share' → copy public URL
5. Paste it in Cell 10 as LANGSMITH_URL
6. Re-run Cells 11 and 15 to regenerate submission.csv

URL format: https://smith.langchain.com/public/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx/r
""")

In [ ]:
# ── Cell 14: Streamlit App reference ─────────────────────────────────────────
print(f"""
🚀 STREAMLIT APP
================
Live URL: {STREAMLIT_URL}
GitHub  : https://github.com/VanjariAbhinay/zyro-hr-bot

Features:
  • BAAI/bge-base-en-v1.5 embeddings
  • MMR retrieval (k=6, fetch_k=20)
  • Multi-query retrieval (3 variants)
  • Dual-layer out-of-scope guardrails
  • LangSmith tracing enabled
  • Source citation display
""")

In [ ]:
# ── Cell 15 (LAST): Generate submission.csv ───────────────────────────────────
if not results:
    print('❌ No results yet!')
    print('   1. Fill in QUESTIONS_ENC in Cell 9 with your encoded questions')
    print('   2. Fill in STREAMLIT_URL and LANGSMITH_URL in Cell 10')
    print('   3. Re-run Cell 11')
    print('   4. Then re-run this cell')
else:
    df = pd.DataFrame(results)
    df = df[['question_id', 'question_enc', 'answer_enc', 'streamlit_link', 'langsmith_link']]
    df.to_csv('submission.csv', index=False)

    print(f'✅ submission.csv generated — {len(df)} rows')
    print()
    print('Submission preview:')
    for _, row in df.iterrows():
        q_text = decode_text(row['question_enc'])
        a_text = decode_text(row['answer_enc'])
        scope  = '❌ OUT-OF-SCOPE (refused)' if 'only answer HR-related' in a_text else '✅ In-scope'
        print(f"  {row['question_id']}: [{scope}] {q_text[:55]}")

    print()
    print('⬇️  Download: File menu → Download → submission.csv')